<a href="https://colab.research.google.com/github/useanynoms-commits/IDRA/blob/main/Assignment12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
"""
Used Car Resale Dataset Preprocessing
"""
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from scipy import stats

# Load dataset
df = pd.read_csv('Day12_Used_Car_Preprocessing_Dataset.csv')

print("Initial Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

print("\nDataset Info:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

# Create copy
df_clean = df.copy()

Initial Shape: (320, 15)

First 5 rows:
    Car_ID       Brand  Year  Mileage_Km  Engine_CC  Power_BHP Fuel_Type  \
0  CAR0001       Skoda  2021       69708       1152      128.8    Diesel   
1  CAR0002      Toyota  2020       88881        903      146.5    Diesel   
2  CAR0003  Volkswagen  2021       43646       1446      185.9    Diesel   
3  CAR0004        Tata  2019       70847       2069      148.8    Petrol   
4  CAR0005        Tata  2016      101228       1657      206.0    Petrol   

  Transmission        City Seller_Type  Condition  Previous_Owners  \
0       Manual     Lucknow  Individual       Good                1   
1    Automatic  Chandigarh  Individual       Good                1   
2    Automatic   Hyderabad  Individual  Very Good                2   
3       Manual     Lucknow  Individual  Excellent                3   
4    Automatic   Ahmedabad      Dealer  Very Good                2   

   Accidents_Reported  Service_Score  Resale_Price_Lakh  
0                   0   

In [10]:
# 1. Handle outliers using Z-score method
print("\n1. HANDLING OUTLIERS (Z-SCORE METHOD)")

numeric_features = ['Year', 'Mileage_Km', 'Engine_CC', 'Power_BHP',
                   'Previous_Owners', 'Accidents_Reported', 'Service_Score', 'Resale_Price_Lakh']

outlier_counts = {}
threshold = 4  # Standard threshold for Z-score

for col in numeric_features:
    z_scores = np.abs(stats.zscore(df_clean[col].dropna()))
    outliers = len(z_scores[z_scores > threshold])
    outlier_counts[col] = outliers
    print(f"{col}: {outliers} outliers")


1. HANDLING OUTLIERS (Z-SCORE METHOD)
Year: 0 outliers
Mileage_Km: 2 outliers
Engine_CC: 4 outliers
Power_BHP: 2 outliers
Previous_Owners: 0 outliers
Accidents_Reported: 0 outliers
Service_Score: 0 outliers
Resale_Price_Lakh: 3 outliers


In [11]:
# Cap outliers
for col in numeric_features:
    z_scores = stats.zscore(df_clean[col])
    mean = df_clean[col].mean()
    std = df_clean[col].std()
    df_clean[col] = df_clean[col].mask(z_scores > threshold, mean + threshold * std)
    df_clean[col] = df_clean[col].mask(z_scores < -threshold, mean - threshold * std)

print("\nOutliers capped")


Outliers capped


In [12]:
# 2. Encode categorical variables
print("\n2. ENCODING CATEGORICAL VARIABLES")

condition_order = {'Poor': 1, 'Fair': 2, 'Good': 3, 'Very Good': 4, 'Excellent': 5}
df_clean['Condition'] = df_clean['Condition'].map(condition_order)
print("Condition: Ordinal")

le_fuel = LabelEncoder()
df_clean['Fuel_Type'] = le_fuel.fit_transform(df_clean['Fuel_Type'])
print("Fuel_Type: Label encoded")

le_trans = LabelEncoder()
df_clean['Transmission'] = le_trans.fit_transform(df_clean['Transmission'])
print("Transmission: Label encoded")

df_clean = pd.get_dummies(df_clean, columns=['Brand', 'City', 'Seller_Type'], drop_first=True)
print("Brand, City, Seller_Type: One-hot encoded")

df_clean = df_clean.drop(columns=['Car_ID'], errors='ignore')
print("Car_ID dropped")

print("\nFinal columns:", len(df_clean.columns))


2. ENCODING CATEGORICAL VARIABLES
Condition: Ordinal
Fuel_Type: Label encoded
Transmission: Label encoded
Brand, City, Seller_Type: One-hot encoded
Car_ID dropped

Final columns: 31


In [13]:
# 3. Separate features and target
print("\n3. FEATURE-TARGET SEPARATION")
X = df_clean.drop('Resale_Price_Lakh', axis=1)
y = df_clean['Resale_Price_Lakh']
print(f"Features: {X.shape[1]}, Target: Resale_Price_Lakh")



3. FEATURE-TARGET SEPARATION
Features: 30, Target: Resale_Price_Lakh


In [14]:
# 4. Train-test split
print("\n4. TRAIN-TEST SPLIT")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")


4. TRAIN-TEST SPLIT
Train: 256, Test: 64


In [15]:
# 5. Feature scaling
print("\n5. FEATURE SCALING")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("StandardScaler fitted on training data only")


5. FEATURE SCALING
StandardScaler fitted on training data only


In [16]:
# 6. Create processed datasets
print("\n6. PROCESSED DATASETS")
train_processed = pd.DataFrame(X_train_scaled, columns=X.columns)
train_processed['Resale_Price_Lakh'] = y_train.values

test_processed = pd.DataFrame(X_test_scaled, columns=X.columns)
test_processed['Resale_Price_Lakh'] = y_test.values

print("Train shape:", train_processed.shape)
print("Test shape:", test_processed.shape)


6. PROCESSED DATASETS
Train shape: (256, 31)
Test shape: (64, 31)


In [17]:
# 7. Export
print("\n7. EXPORTING")
train_processed.to_csv('Train_Day_12_Dataset.csv', index=False)
test_processed.to_csv('Test_Day_12_Dataset.csv', index=False)
print("Saved as 'Train_Day_12_Dataset.csv' and 'Test_Day_12_Dataset.csv'")


7. EXPORTING
Saved as 'Train_Day_12_Dataset.csv' and 'Test_Day_12_Dataset.csv'
